# P04 · Notebook 31 · The four simple families on DCASE 2020 Task 2

**Why.** The deck presents five families of detectors. Families 1 to 4 were only tried on four pumps; family 5 alone went to the public
tests. This notebook runs families 1 to 4 on DCASE 2020, with the settings of the pump runs unchanged, so that all five can be set
side by side on the same test. The rules were written first: `results/mimii_beats/RULES_simple_families_written_before_running.md`.

**What it does.** For every machine (a machine ID of a type): fit each family on that unit's normal training clips, score its test clips, and compute
mean AUC over the six machine types exactly as the official evaluator does. The evaluation labels are read once, at the end, after every score is saved.

| | Family | What it reads | Detector |
|---|---|---|---|
| 1 | A limit on one number | the clip's loudness | distance from the usual loudness |
| 2 | Distance from the normal cloud | 128 numbers per clip | PCA, T² + SPE |
| 3 | Isolation | 128 numbers per clip | Isolation Forest |
| 4 | Autoencoder | the log-mel picture, 5 slices at a time | reconstruction error |

**How to run.** Runtime › Run all. **No GPU is needed** (a CPU runtime is enough). About 12 gb of downloads from zenodo (the 18 zips of notebooks 25 and 27, and the label file); a zip is deleted once its
clips are stored. Expected time: **about 2 to 3 hours** (an estimate: one network took 2.2 minutes on a 2-CPU test machine; Colab may be slower). Almost all of it is family 4, which trains one small network per unit
(41 units). Every finished unit is saved in Drive › `P04_SimpleFamilies_2020`, so if the session stops, **Run all again continues** where it stopped
(the downloads are repeated, the trained units are not).

**Send back** the zip that downloads at the end (`p04_n31_results.zip`).

In [ ]:
# 1 · Setup (no GPU is needed: a CPU runtime is enough)
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, math, gc, csv, warnings
from pathlib import Path
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT, WORK = Path("/content/drive/MyDrive") / "P04_SimpleFamilies_2020", Path("/content/work")
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_SimpleFamilies_2020")), Path(os.environ.get("P04_WORK", "work"))
for p in (OUT, WORK, OUT / "scores"):
    p.mkdir(parents=True, exist_ok=True)
T_START = time.time()
print("free local disk", round(shutil.disk_usage(WORK).free / 1e9), "GB · free Drive", round(shutil.disk_usage(OUT).free / 1e9, 1), "GB · CPUs", os.cpu_count())


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()


download_one = download                              # notebook 11's single-connection download


def download(url, dest):
    # several connections per file with aria2c (notebook 19 saw about 2.4 MB/s per connection from Zenodo); falls back to one connection
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if not url.startswith("file://"):
        if shutil.which("aria2c") is None and IN_COLAB:
            if subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True).returncode != 0:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
        if shutil.which("aria2c"):
            r = subprocess.run(["aria2c", "-x", "4", "-s", "4", "-k", "8M", "--file-allocation=none", "--max-tries=30", "--retry-wait=10",
                                "--allow-overwrite=true", "--auto-file-renaming=false", "--console-log-level=warn", "--summary-interval=0",
                                "--download-result=hide", "-d", str(dest.parent), "-o", part.name, url])
            if r.returncode == 0 and part.exists():
                part.rename(dest)
                return
            print(f"  aria2c failed for {dest.name} (code {r.returncode}); one connection instead", flush=True)
            for f in (part, Path(str(part) + ".aria2")):
                f.unlink(missing_ok=True)
    download_one(url, dest)

In [ ]:
# 2 · The files, and what was fixed in the rules before this notebook ran
RULES = "results/mimii_beats/RULES_simple_families_written_before_running.md"
# DCASE 2020 Task 2: the file lists and MD5s of notebooks 25 and 27 (the Zenodo records)
ZD, ZX, ZE, ZG = ("https://zenodo.org/records/3678171/files/", "https://zenodo.org/records/3727685/files/",
                  "https://zenodo.org/records/3841772/files/", "https://zenodo.org/records/3951620/files/")
DMD5 = {"fan": "649bdfc06263ae7a838963f43b6641e6", "pump": "90e7091ef722b7238a7f1009365779cd", "slider": "da24a757719f0d94d5aa2d646bbfdc86", "valve": "34d2672f55bb041589ef79c10dc89934", "ToyCar": "4dec75ca8d9f666aa9e4c1894a740501", "ToyConveyor": "03b6aa1bfd09a39d53af0ba39f71fa91"}
XMD5 = {"fan": "567798854130c8019df8b664c095ed1e", "pump": "4e33ae9c0db5cc88437675f5e317caee", "slider": "f84f186f4295fc23754abcdd1b0580a7", "valve": "057cdd560caffb8305449541afd28c6d", "ToyCar": "a5d4ccd498af70b04dd74b644b820006", "ToyConveyor": "0e808ac0b761e9fb13d5e0c65caff00d"}
EMD5 = {"fan": "1eb9356a768cadfd0f2e59a5c57e578b", "pump": "23a8f8f924be218c69df67fe07360348", "slider": "0193c769073840332ce3aad84b1ccaa2", "valve": "f69c551d2088691050d20cccca9d631c", "ToyCar": "bea2bdd612f616be8f2b8eb087b32c7c", "ToyConveyor": "ccfcc9847c7d3404d7ee9d6d9e0b2ba2"}
DC_DEV = {t_: (ZD + f"dev_data_{t_}.zip?download=1", f"dev_data_{t_}.zip", h) for t_, h in DMD5.items()}
XZIPS = {t_: (ZX + f"eval_data_train_{t_}.zip?download=1", f"eval_data_train_{t_}.zip", h) for t_, h in XMD5.items()}
DC_EVAL = {t_: (ZE + f"eval_data_test_{t_}.zip?download=1", f"eval_data_test_{t_}.zip", h) for t_, h in EMD5.items()}
DC_GT = (ZG + "eval_data_list.csv?download=1", "eval_data_list.csv", "f16aaa185df9906e588864aaf7cb58ea")
TYPES = ["fan", "pump", "slider", "valve", "ToyCar", "ToyConveyor"]
DEV_IDS = {"fan": ["id_00", "id_02", "id_04", "id_06"], "pump": ["id_00", "id_02", "id_04", "id_06"], "slider": ["id_00", "id_02", "id_04", "id_06"], "valve": ["id_00", "id_02", "id_04", "id_06"], "ToyCar": ["id_01", "id_02", "id_03", "id_04"], "ToyConveyor": ["id_01", "id_02", "id_03"]}
EVAL_IDS = {"fan": ["id_01", "id_03", "id_05"], "pump": ["id_01", "id_03", "id_05"], "slider": ["id_01", "id_03", "id_05"], "valve": ["id_01", "id_03", "id_05"], "ToyCar": ["id_05", "id_06", "id_07"], "ToyConveyor": ["id_04", "id_05", "id_06"]}
SR = 16000
NS_RULE = 10 * SR                                                     # the first 10 s of a clip (ToyCar's clips are about 11 s)
# what the kept system (family 5) and the others reached on the evaluation set, from this project's records and the official tables
RECORDED = {"Family 5 · the kept system (expert + trainee)": 0.9747, "rank-1 team": 0.9458}
RECORDED_NOTE = "mean AUC over the six machine types, evaluation set"
if TEST:
    _dc = json.loads(os.environ["P04_TEST_DCASE"]); _c = json.loads(os.environ.get("P04_TEST_CFG", "{}"))
    DC_DEV, XZIPS, DC_EVAL = ({t_: tuple(v) for t_, v in _dc[k_].items()} for k_ in ("dev", "add", "eval"))
    DC_GT, TYPES, DEV_IDS, EVAL_IDS = tuple(_dc["gt"]), list(_dc["dev"]), _dc["dev_ids"], _dc["eval_ids"]
    NS_RULE = _c.get("clip_samples", NS_RULE)

# the four simple families, with the settings of the pump runs (results/mimii/RULES_written_before_running.md), unchanged
FAMS = ["F1", "F2", "F3", "F4"]
FAM_NAME = {"F1": "Family 1 · a limit on one number (loudness)", "F2": "Family 2 · distance from the normal cloud (128 clip statistics)",
            "F3": "Family 3 · isolation (Isolation Forest, 128 clip statistics)", "F4": "Family 4 · autoencoder (reconstruction error)",
            "F1_raw": "reported beside · loudness as it is (louder = more anomalous)"}
SCORED = FAMS + ["F1_raw"]
N_FFT, HOP, N_MELS = 1024, 512, 64

In [ ]:
# 3 · The data cell of the notebook that ran on these files (its readers and its loader, unchanged)
# 9 · DCASE 2020's own files of the two types (MD5 against the Zenodo records), and every clip's first 10 s in one int16 store on the
# local disk; gates G0(a) and G0(b). The readers and the loader are notebook 23's (through notebook 25); the loader has these types'
# machine IDs, the cut to 10 s, and a check of every file.
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res


def to_int16(v):
    # only for microphone-0 samples read from the 16-bit WAV files, so this conversion is exact
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def f32(a):
    # a slice of a stream → float32 signal: int16 streams hold samples × 32768, float16 streams hold the signal itself
    a = np.asarray(a)
    return a.astype(np.float32) / 32768.0 if a.dtype == np.int16 else a.astype(np.float32)


STORE = WORK / "stores27"; STORE.mkdir(exist_ok=True)
DST, DMETA = {}, {}


def dcase_load(t):
    # every DCASE 2020 clip of type t in one int16 store, cut to its first NS_RULE samples (10 s, as the view's reference code cuts
    # it): development train and test, the additional training clips (all of them) and the evaluation test clips
    meta_f, arr_f = STORE / f"dc_{t}.json", STORE / f"dc_{t}.npy"
    if meta_f.exists() and arr_f.exists():
        DMETA[t] = json.loads(meta_f.read_text()); DST[t] = np.load(arr_f, mmap_mode="r")
        return
    zfs = {"dev": zipfile.ZipFile(get_file(DC_DEV[t])), "evaltrain": zipfile.ZipFile(get_file(XZIPS[t])), "evaltest": zipfile.ZipFile(get_file(DC_EVAL[t]))}
    rows, skipped = [], []
    for kind, zf in zfs.items():
        for nm in sorted(x for x in zf.namelist() if x.lower().endswith(".wav")):
            parts = nm.replace("\\", "/").split("/"); base, folder = parts[-1], parts[-2]
            if "__MACOSX" in parts or base.startswith("._"):          # an archive's metadata entry, not a clip (none is expected)
                skipped.append(nm); continue
            part = "train" if folder == "train" else "test"
            label = (1 if base.startswith("anomaly") else 0) if (kind == "dev" and part == "test") else (0 if part == "train" else -1)
            rows.append((kind, nm, "dev" if kind == "dev" else "eval", part, re.search(r"(id_\d+)", base).group(1), label, base))
    ids = {(s_, p_): sorted(set(r[4] for r in rows if r[2] == s_ and r[3] == p_)) for s_ in ("dev", "eval") for p_ in ("train", "test")}
    if not (ids[("dev", "train")] == ids[("dev", "test")] == DEV_IDS[t] and ids[("eval", "train")] == ids[("eval", "test")] == EVAL_IDS[t]):
        raise SystemExit(f"stop: G0(b) failed. The machine IDs of {t} are not the rules': {ids}")
    assert not any(r[2] == "eval" and r[3] == "train" and not r[6].startswith("normal") for r in rows), "the additional training clips are normal"
    ns, lengths = NS_RULE, Counter()
    tmp = arr_f.with_name(arr_f.stem + ".part.npy")
    mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.int16, shape=(len(rows), ns))
    for i, (x, sr) in enumerate(stream(lambda i: read_all(zfs[rows[i][0]].read(rows[i][1])), range(len(rows)))):
        if not (sr == SR and x.shape[1] == 1 and x.shape[0] >= ns):
            raise SystemExit(f"stop: G0(b) failed. {rows[i][1]} is not a 16 kHz, one-channel clip of at least {ns / SR:g} s "
                             f"(rate {sr}, channels {x.shape[1]}, samples {x.shape[0]})")
        lengths[(f"{rows[i][2]}|{rows[i][3]}", int(x.shape[0]))] += 1
        mm[i] = to_int16(x[:ns, 0])
    mm.flush(); del mm
    os.replace(tmp, arr_f)
    for zf in zfs.values():
        zf.close()
    DMETA[t] = {"path": [r[1] for r in rows], "name": [r[6] for r in rows], "set": [r[2] for r in rows], "part": [r[3] for r in rows],
                "mid": [r[4] for r in rows], "label": [r[5] for r in rows], "samples": int(ns), "skipped_archive_entries": skipped,
                "lengths": {k: {str(n_): int(c) for (k2, n_), c in sorted(lengths.items()) if k2 == k} for k in sorted(set(k_ for k_, _ in lengths))},
                "zips": {"dev": list(DC_DEV[t][1:]), "additional": list(XZIPS[t][1:]), "evaluation": list(DC_EVAL[t][1:])}}
    meta_f.write_text(json.dumps(DMETA[t]))
    DST[t] = np.load(arr_f, mmap_mode="r")
    if IN_COLAB:
        for z in (DC_DEV[t], XZIPS[t], DC_EVAL[t]):
            (WORK / z[1]).unlink(missing_ok=True)


def dc_rows(t, set_, part, q=None):
    m = DMETA[t]
    return np.array([i for i in range(len(m["name"])) if m["set"][i] == set_ and m["part"][i] == part and (q is None or m["mid"][i] == q)], dtype=np.int64)


t_data = time.time()
DCASE = {"counts": {}, "lengths": {}}
for t in TYPES:
    dcase_load(t)
    DCASE["counts"][t] = {f"{s_}|{p_}": {q: int(len(dc_rows(t, s_, p_, q))) for q in sorted(set(np.array(DMETA[t]["mid"])[dc_rows(t, s_, p_)].tolist()))}
                          for s_ in ("dev", "eval") for p_ in ("train", "test")}
    DCASE["lengths"][t] = DMETA[t]["lengths"]
    print(f"DCASE 2020 · {t}: {len(DMETA[t]['name'])} clips {DCASE['counts'][t]} · samples per clip before the cut (set|part: samples: clips): "
          f"{DCASE['lengths'][t]}", flush=True)
NS = int(DMETA[TYPES[0]]["samples"])
assert all(int(DMETA[t]["samples"]) == NS for t in TYPES) and NS == NS_RULE, "stop: a store on the local disk was not cut to the rules' length"

# G0(a) · every zip is the published file: get_file stops on another MD5, and a store keeps the names and MD5s of the zips it was built from
_want = {t: {"dev": list(DC_DEV[t][1:]), "additional": list(XZIPS[t][1:]), "evaluation": list(DC_EVAL[t][1:])} for t in TYPES}
G0 = {"a": {"ok": bool(all(DMETA[t]["zips"] == _want[t] for t in TYPES)), "zips": _want}}
print("G0(a) · the six zips are the published files (MD5):", G0["a"]["ok"])
if not G0["a"]["ok"]:
    raise SystemExit(f"stop: G0(a) failed. A store in {STORE} was built from other files; delete that folder and run again")


def f32(a):
    return np.asarray(a).astype(np.float32) / 32768.0


print(f"data ready · {sum(len(DMETA[t]['name']) for t in TYPES)} clips · {NS / SR:g} s each · {(time.time() - t_data) / 60:.1f} min")

In [ ]:
# 4 · The log-mel picture of every clip, with the code of the pump runs (code/p04_mimii_features.py, unchanged): 64 bands,
# slices of 1,024 readings, a new slice every 512. One float16 file per machine type on the local disk; a finished type is not redone
EPS = sys.float_info.epsilon


def hz_to_mel(f):
    """Slaney mel scale (librosa htk=False)."""
    f = np.asanyarray(f, dtype=float)
    f_sp = 200.0 / 3; mels = f / f_sp
    min_log_hz = 1000.0; min_log_mel = min_log_hz / f_sp; logstep = np.log(6.4) / 27.0
    return np.where(f >= min_log_hz, min_log_mel + np.log(np.maximum(f, 1e-10) / min_log_hz) / logstep, mels)


def mel_to_hz(m):
    m = np.asanyarray(m, dtype=float)
    f_sp = 200.0 / 3; freqs = f_sp * m
    min_log_hz = 1000.0; min_log_mel = min_log_hz / f_sp; logstep = np.log(6.4) / 27.0
    return np.where(m >= min_log_mel, min_log_hz * np.exp(logstep * (m - min_log_mel)), freqs)


def mel_filters(sr=SR, n_fft=N_FFT, n_mels=N_MELS):
    """Triangular filters with Slaney area normalisation (librosa.filters.mel defaults)."""
    fftfreqs = np.fft.rfftfreq(n_fft, 1.0 / sr)
    mel_f = mel_to_hz(np.linspace(hz_to_mel(0.0), hz_to_mel(sr / 2.0), n_mels + 2))
    fdiff = np.diff(mel_f); ramps = mel_f[:, None] - fftfreqs[None, :]
    w = np.zeros((n_mels, len(fftfreqs)))
    for i in range(n_mels):
        lower = -ramps[i] / fdiff[i]; upper = ramps[i + 2] / fdiff[i + 1]
        w[i] = np.maximum(0, np.minimum(lower, upper))
    w *= (2.0 / (mel_f[2:n_mels + 2] - mel_f[:n_mels]))[:, None]
    return w


def power_spec(y):
    """|STFT|² with centred frames and zero padding (librosa ≥ 0.10 default pad_mode='constant')."""
    y = np.pad(y, (N_FFT // 2, N_FFT // 2), mode="constant")
    n = 1 + (len(y) - N_FFT) // HOP
    idx = np.arange(N_FFT)[None, :] + HOP * np.arange(n)[:, None]
    return (np.abs(np.fft.rfft(y[idx] * WIN, axis=1)) ** 2).T


MEL = mel_filters(SR, N_FFT, N_MELS)
WIN = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(N_FFT) / N_FFT)       # periodic Hann, as in the pump runs
N_FRAMES = 1 + NS // HOP
FEATS = WORK / "logmel"; FEATS.mkdir(exist_ok=True)
LM = {}


def logmel_of(i_t):
    i, t = i_t
    return (10.0 * np.log10(MEL @ power_spec(f32(DST[t][i])) + EPS)).astype(np.float16)


t_feat = time.time()
for t in TYPES:
    f, n = FEATS / f"lm_{t}.npy", len(DMETA[t]["name"])
    if not f.exists():
        tmp = f.with_name(f.stem + ".part.npy")
        mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(n, N_MELS, N_FRAMES))
        for i, lm in enumerate(stream(logmel_of, [(i, t) for i in range(n)], depth=16, workers=max(2, (os.cpu_count() or 2)))):
            mm[i] = lm
        mm.flush(); del mm
        os.replace(tmp, f)
    LM[t] = np.load(f, mmap_mode="r")
    assert LM[t].shape == (n, N_MELS, N_FRAMES)
    print(f"{t}: {n} clips → log-mel {LM[t].shape} · {(time.time() - t_feat) / 60:.1f} min so far", flush=True)
FEAT_S = round(time.time() - t_feat, 1)


In [ ]:
# 5 · The four families (code/p04_mimii_models.py, unchanged classes) and the units they are fitted on. A unit is one machine (a machine ID of a type):
# each family is fitted on that unit's normal training clips only, and scores that unit's test clips. No label is read here
def frames5(lm):
    """(64, T) log-mel → (T-4, 320): five consecutive frames side by side, as the baseline."""
    T = lm.shape[1]
    return np.concatenate([lm[:, t:T - 4 + t].T for t in range(5)], axis=1)


def clip_stats(logmel):
    x = logmel.astype(np.float32)
    return np.c_[x.mean(2), x.std(2)]


class AEfam:
    """M1, set up like the Keras baseline: no L2 penalty (Keras adds none), and the last 10 % of the
    training rows left out (Keras validation_split=0.1 holds them out before shuffling).
    Chosen on 2026-11-06 after the inspection (results/mimii/DECISION_after_inspection.md). It is also
    about five times faster: scikit-learn's default L2 slowly pushes dead-unit weights into float32
    denormals. The first run (alpha 0.0001, all rows) is kept in results/mimii/first_run/."""
    ALPHA, KEEP = 0.0, 0.9

    def fit(self, LM):
        X = np.concatenate([frames5(m.astype(np.float32)) for m in LM])
        X = X[: int(len(X) * self.KEEP)]                  # Keras: split_at = int(len(x) * (1 - validation_split))
        self.m = MLPRegressor(hidden_layer_sizes=(64, 64, 8, 64, 64), activation="relu", solver="adam", alpha=self.ALPHA,
                              batch_size=512, max_iter=50, n_iter_no_change=50, tol=0, shuffle=True,
                              random_state=0).fit(X, X)
        return self

    def score(self, LM):
        out = []
        for m in LM:
            X = frames5(m.astype(np.float32))
            out.append(float(np.mean((self.m.predict(X) - X) ** 2)))
        return np.array(out)


class Maha:
    """Standardise → PCA (keep 95 % or all) → T² + SPE, each over its 99th training percentile."""
    def __init__(self, var=0.95):
        self.var = var

    def fit(self, X):
        self.sc = StandardScaler().fit(X); Z = self.sc.transform(X)
        p = PCA().fit(Z)
        k = len(p.explained_variance_) if self.var is None else int(np.searchsorted(np.cumsum(p.explained_variance_ratio_), self.var) + 1)
        self.P, self.lam = p.components_[:k].T, p.explained_variance_[:k]
        t2, spe = self._parts(Z)
        self.qt, self.qs = np.quantile(t2, 0.99), max(np.quantile(spe, 0.99), 1e-12)
        return self

    def _parts(self, Z):
        T = Z @ self.P
        return (T ** 2 / self.lam).sum(1), ((Z - T @ self.P.T) ** 2).sum(1)

    def score(self, X):
        t2, spe = self._parts(self.sc.transform(X))
        return t2 / self.qt + spe / self.qs


class IFfam:
    def fit(self, X):
        self.sc = StandardScaler().fit(X)
        self.m = IsolationForest(n_estimators=200, random_state=0).fit(self.sc.transform(X))
        return self

    def score(self, X):
        return -self.m.score_samples(self.sc.transform(X))


def units(set_):
    return [(t, q) for t in TYPES for q in (DEV_IDS if set_ == "dev" else EVAL_IDS)[t]]


ALL_UNITS = [(s_, t, q) for s_ in ("dev", "eval") for t, q in units(s_)]


def unit_file(s_, t, q):
    return OUT / "scores" / f"{s_}_{t}_{q}.npz"


def load_unit(s_, t, q):
    f = unit_file(s_, t, q)
    return dict(np.load(f, allow_pickle=False)) if f.exists() else {}


def save_unit(s_, t, q, d):
    f = unit_file(s_, t, q); tmp = f.with_name(f.stem + ".part.npz")
    np.savez(tmp, **d); os.replace(tmp, f)


def fit_fast(s_, t, q):
    # families 1, 2 and 3: seconds per unit
    tr, te = dc_rows(t, s_, "train", q), dc_rows(t, s_, "test", q)
    A, B = np.asarray(LM[t][tr], dtype=np.float32), np.asarray(LM[t][te], dtype=np.float32)
    la, lb = A.mean((1, 2)), B.mean((1, 2))                           # one number per clip: its overall loudness, as on the pump page
    SA, SB = clip_stats(A), clip_stats(B)                             # 128 numbers per clip: the average and the wobble of each band
    return {"name": np.array(DMETA[t]["name"])[te], "F1": np.abs(lb - la.mean()) / max(float(la.std()), 1e-9), "F1_raw": lb,
            "F2": Maha(0.95).fit(SA).score(SB), "F3": IFfam().fit(SA).score(SB), "n_train": np.array(len(tr))}


def fit_ae(s_, t, q):
    tr, te = dc_rows(t, s_, "train", q), dc_rows(t, s_, "test", q)
    return AEfam().fit(np.asarray(LM[t][tr])).score(np.asarray(LM[t][te]))


t_fast = time.time()
for s_, t, q in ALL_UNITS:
    d = load_unit(s_, t, q)
    if "F3" not in d:
        save_unit(s_, t, q, {**d, **fit_fast(s_, t, q)})
FAST_S = round(time.time() - t_fast, 1)
print(f"families 1, 2 and 3: {len(ALL_UNITS)} units fitted and scored · {FAST_S / 60:.1f} min")


In [ ]:
# 6 · The official numbers of DCASE 2020: per machine, AUC and pAUC (max_fpr 0.1); per type, their means over its machines; and the
# mean over the types. dc_metrics is notebook 27's function, whose numbers the official evaluator reproduced there
def dc_metrics(y, s):
    # as the official evaluator: sklearn's AUC, and its pAUC with max_fpr = 0.1 (standardised)
    return {"AUC": float(roc_auc_score(y, s)), "pAUC": float(roc_auc_score(y, s, max_fpr=0.1)), "n_test": int(len(y)), "n_anomalies": int(np.sum(y))}


def table(set_, labels):
    # labels: (type, machine) → array of 0/1 in the order of that unit's saved names
    out = {}
    for f in SCORED:
        per = {}
        for t, q in units(set_):
            d = load_unit(set_, t, q)
            if f in d:
                per.setdefault(t, {})[q] = dc_metrics(labels[(t, q)], d[f])
        if len(per) == len(TYPES) and all(len(per[t]) == len((DEV_IDS if set_ == "dev" else EVAL_IDS)[t]) for t in TYPES):
            ty = {t: {"AUC": float(np.mean([v["AUC"] for v in per[t].values()])), "pAUC": float(np.mean([v["pAUC"] for v in per[t].values()])),
                      "per_machine": per[t]} for t in TYPES}
            out[f] = {"score": float(np.mean([ty[t]["AUC"] for t in TYPES])), "mean_pAUC": float(np.mean([ty[t]["pAUC"] for t in TYPES])), "per_type": ty}
    return out


def show(res, title):
    print(title); print(f"  {'':<62}{'mean AUC':>10}{'mean pAUC':>11}   " + "".join(f"{t[:11]:>12}" for t in TYPES))
    for f in [f_ for f_ in SCORED if f_ in res]:
        print(f"  {FAM_NAME[f]:<62}{res[f]['score']:>10.4f}{res[f]['mean_pAUC']:>11.4f}   " + "".join(f"{res[f]['per_type'][t]['AUC']:>12.4f}" for t in TYPES))


def dev_labels():
    return {(t, q): np.array(DMETA[t]["label"])[dc_rows(t, "dev", "test", q)] for t, q in units("dev")}


_m = dc_metrics([0, 0, 1, 1], [0.1, 0.4, 0.35, 0.8]); assert abs(_m["AUC"] - 0.75) < 1e-12
DEV = table("dev", dev_labels())
show(DEV, "DCASE 2020 · development set · families 1 to 3 (family 4 follows in the next cell)")


In [ ]:
# 7 · Family 4, the autoencoder: one network per unit (the slow part). Each finished unit is saved in Drive › P04_SimpleFamilies_2020 › scores, so a
# session that stops is continued by Runtime › Run all
t_ae, done_now = time.time(), 0
todo = [u for u in ALL_UNITS if "F4" not in load_unit(*u)]
print(f"autoencoders to train: {len(todo)} of {len(ALL_UNITS)} units")
for k, (s_, t, q) in enumerate(todo):
    t1 = time.time(); d = load_unit(s_, t, q); d["F4"] = fit_ae(s_, t, q); d["F4_seconds"] = np.array(round(time.time() - t1, 1))
    save_unit(s_, t, q, d); done_now += 1
    left = (time.time() - t_ae) / done_now * (len(todo) - done_now)
    print(f"  {k + 1}/{len(todo)} · {s_} {t} {q}: {time.time() - t1:.0f} s · about {left / 60:.0f} min left", flush=True)
AE_S = round(sum(float(load_unit(*u)["F4_seconds"]) for u in ALL_UNITS), 1)
print(f"family 4: all {len(ALL_UNITS)} units done · training time in total {AE_S / 60:.0f} min")


In [ ]:
# 8 · The development set with all four families, then the evaluation set, scored once. Every score file is already written (cells 5
# and 7) and is locked here before a single evaluation label is read
assert all(all(f in load_unit(*u) for f in SCORED) for u in ALL_UNITS), "stop: a unit has no scores yet; run the cells above"
DEV = table("dev", dev_labels())
show(DEV, "DCASE 2020 · development set · the four simple families")
LOCK = OUT / "scores_written.json"
_sha = {f"{s_}_{t}_{q}": hashlib.sha256(unit_file(s_, t, q).read_bytes()).hexdigest() for s_, t, q in ALL_UNITS}
if LOCK.exists():
    assert json.loads(LOCK.read_text())["sha256"] == _sha, "stop: the score files changed after they were locked"
else:
    LOCK.write_text(json.dumps({"written": time.strftime("%Y-%m-%d %H:%M"), "sha256": _sha}, indent=1))


def read_labels():
    # the ground truth of the two types: (type, file) → label, and the file's lines of these two sections, unchanged
    lab, lines, types_in_file, cur = {}, [], [], None
    with open(get_file(DC_GT), newline="") as fh:                     # stops unless the file's MD5 is the Zenodo record's
        for line in fh:
            row = next(csv.reader([line]), [])
            if not row:
                continue
            if len(row) < 2:                                          # a machine-type line, as the official evaluator reads it
                cur = row[0].strip(); types_in_file.append(cur)
            elif cur in TYPES:
                lab[(cur, row[0].strip())] = int(float(row[2]))
            if cur in TYPES:
                lines.append(line)
    return lab, lines, types_in_file


def eval_labels():
    lab, _, _ = read_labels(); out = {}
    for t, q in units("eval"):
        names = np.array(DMETA[t]["name"])[dc_rows(t, "eval", "test", q)]
        assert all((t, n) in lab for n in names), f"{t} {q}: a clip is not in the ground truth"
        out[(t, q)] = np.array([lab[(t, n)] for n in names])
    return out


EVAL = table("eval", eval_labels())                                  # the evaluation labels: read only here
show(EVAL, "DCASE 2020 · evaluation set · the four simple families")
print("\nbeside them, from the records (" + RECORDED_NOTE + "):")
for k, v in RECORDED.items():
    print(f"  {k:<62}{v:>10.4f}")


In [ ]:
# 9 · Save, and one zip to download
BEST = max(FAMS, key=lambda f: EVAL[f]["score"]); F5 = RECORDED["Family 5 · the kept system (expert + trainee)"]
RESULT = {"rules": RULES, "notebook": 31, "year": 2020, "date": time.strftime("%Y-%m-%d %H:%M"), "test_mode": TEST, "score": "mean AUC over the six machine types",
          "families": FAM_NAME, "settings": {"n_fft": N_FFT, "hop": HOP, "n_mels": N_MELS, "frames": int(N_FRAMES), "clip_samples": int(NS),
                                             "F2": "StandardScaler → PCA 95 % → T² + SPE", "F3": "IsolationForest, 200 trees, seed 0",
                                             "F4": "MLP 320-64-64-8-64-64-320, 50 epochs, batch 512, seed 0, first 90 % of the rows"},
          "units": {"dev": len(units("dev")), "eval": len(units("eval"))}, "development": DEV, "evaluation": EVAL, "recorded_beside": RECORDED,
          "recorded_note": RECORDED_NOTE,
          "answer": {"best_simple_family": BEST, "its_score": round(EVAL[BEST]["score"], 4), "family_5_kept_system": F5,
                     "family_5_less_best_simple": round(F5 - round(EVAL[BEST]["score"], 4), 4),
                     "a_simple_family_reaches_family_5": bool(round(EVAL[BEST]["score"], 4) >= F5)},
          "clips": {t: len(DMETA[t]["name"]) for t in TYPES}, "zips": {t: DMETA[t]["zips"] for t in TYPES},
          "seconds": {"log_mel": FEAT_S, "families_1_2_3": FAST_S, "family_4": AE_S, "session": round(time.time() - T_START, 1)}}
(OUT / "simple_families_2020.json").write_text(json.dumps(RESULT, indent=1))
print(json.dumps(RESULT["answer"], indent=1))
Z = OUT / "p04_n31_results.zip"
with zipfile.ZipFile(Z, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT / "simple_families_2020.json", "simple_families_2020.json"); zf.write(LOCK, LOCK.name)
    for u in ALL_UNITS:
        zf.write(unit_file(*u), f"scores/{unit_file(*u).name}")
print("saved:", Z, f"({Z.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(Z))


## Send back
Send the zip that downloaded (`p04_n31_results.zip`). It is also in Drive › `P04_SimpleFamilies_2020`.